In [2]:
from typing import List, Dict, Any, Optional, Set
from pydantic import BaseModel
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.schema import Document
from langchain.chains import RetrievalQA
from datetime import datetime
import psycopg2
import psycopg2.extras
from psycopg2.extras import Json
import os
import json
import logging
import re


In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
VECTOR_STORE_DIR = os.getenv("VECTOR_STORE_DIR", "chroma_collection_setup")
LINEAGE_CSV_PATH = os.getenv("LINEAGE_CSV_PATH", "temp_lineage_data/lineage_output_deep.csv")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
DATABASE_URL = os.getenv("DATABASE_URL")

In [6]:
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY)
LLM = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=GOOGLE_API_KEY, temperature=0.2)

E0000 00:00:1758546938.790141 5674641 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.
E0000 00:00:1758546938.802519 5674641 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [3]:
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import CSVLoader
from langchain.chains import RetrievalQA


def get_org_vector_store(org_id: str) -> Chroma:
    """
    Returns a Chroma vector store bound to a specific org collection.
    """
    collection_name = f"org_{org_id}"
    db = Chroma(
        collection_name=collection_name,
        persist_directory=VECTOR_STORE_DIR,
        embedding_function=embedding,
    )
    return db


def init_org_vector_store(org_id: str, csv_path: str = None) -> Chroma:
    """
    Initialize or update an org-specific collection.
    Optionally bootstrap with CSV data.
    """
    db = get_org_vector_store(org_id)

    if csv_path:  # bootstrap docs into the org collection
        loader = CSVLoader(csv_path)
        docs = loader.load()
        db.add_documents(docs)
        db.persist()
        print(f"Loaded {len(docs)} docs into collection for org {org_id}")

    return db


def get_retriever(org_id: str, k: int = 8):
    db = get_org_vector_store(org_id)
    return db.as_retriever(search_kwargs={"k": k})


def get_qa_chain(org_id: str, k: int = 5):
    retriever = get_retriever(org_id, k=k)
    qa_chain = RetrievalQA.from_chain_type(
        llm=LLM,
        retriever=retriever,
        return_source_documents=True,
    )
    return qa_chain

In [5]:
# Initialize (bootstrap) a vector store for org_123 with CSV data
DB = init_org_vector_store("76d33fb3-6062-456b-a211-4aec9971f8be", LINEAGE_CSV_PATH)

/var/folders/d3/960_llfs0q11myz6l62g1y140000gn/T/ipykernel_99090/3281543355.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(


Loaded 33 docs into collection for org 76d33fb3-6062-456b-a211-4aec9971f8be


/var/folders/d3/960_llfs0q11myz6l62g1y140000gn/T/ipykernel_99090/3281543355.py:30: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [7]:
# # Initialize (bootstrap) a vector store for org_123 with CSV data
# DB = init_org_vector_store("123_org_1", LINEAGE_CSV_PATH)

# Create retriever for that org
RETRIEVER = get_retriever("76d33fb3-6062-456b-a211-4aec9971f8be", k=8)

# Create QA chain for that org
qa_chain = get_qa_chain("76d33fb3-6062-456b-a211-4aec9971f8be", k=5)

# Run a query
result = qa_chain.invoke({"query": "need to get the downstream impact of prod_tz.edw_staging.allocationrule,payrollbasis"})
print(result)


/var/folders/d3/960_llfs0q11myz6l62g1y140000gn/T/ipykernel_38082/3281543355.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(


{'query': 'need to get the downstream impact of prod_tz.edw_staging.allocationrule,payrollbasis', 'result': 'The column `prod_tz.edw_staging.allocationrule.payrollbasis` has a direct impact on `prod_tz.edw_staging_edw.dimsharedservicesallocationrule.payrollbasis`.\nThe column `prod_tz.edw_staging_edw.dimsharedservicesallocationrule.payrollbasis` has a direct impact on `prod_tz.edw_marts.martfinanceallocation.payrollbasis`.\nThe column `prod_tz.edw_reporting.rptsharedservicesallocation.payrollbasis` has a direct impact on `prod_tz.edw_reporting.rptcostallocation.payrollbasis`.\nThe column `prod_tz.edw_analytics.analyssharedallocation.payrollbasis` has a direct impact on `prod_tz.edw_analytics.analyzcostallocation.payrollbasis`.', 'source_documents': [Document(metadata={'source': 'temp_lineage_data/lineage_output_deep.csv', 'row': 3}, page_content="source_database: prod_tz\nsource_schema: edw_staging\nsource_table: allocationrule\nsource_column: payrollbasis\ntarget_database: prod_tz\nta